# The season simulator: the rest of the Bundesliga, played 50,000 times

Companion notebook to the [Season page](https://jhvl88.github.io/matchday/season/) and the [season simulator report](https://github.com/JHvL88/matchday/blob/main/analysis/backtest/season/README.md) of [Jan's Matchday](https://jhvl88.github.io/matchday/). Everything here runs from public data, on a CPU, in a few minutes; no GPU needed.

1. **The published run**: replay the site's numbers from the ratings in `data/season.json`.
2. **Ratings from scratch**: fit both sets of ratings from football-data.co.uk and simulate the season as it stands today.
3. **What if?**: lock in some results and play the rest.
4. **The test**: the pre-registered backtest's numbers, and one test season recomputed to check them.

One thing needs the private data repository: my models' ratings learn from Understat's xG. Without it they learn from goals only, so part 2's "models" numbers differ a little from the site's. The bookmakers' ratings need nothing private and reproduce exactly.

In [ ]:
LATEST = False         # True: today's source files; False: the ones the report used
import json, os, subprocess, sys
REPO = "https://github.com/JHvL88/matchday"
BRANCH = os.environ.get("MATCHDAY_BRANCH", "main")
SOURCE_COMMIT = "08f400c56187a033afd7bc4d50a8e78a780bd154"     # source-data commit behind the report

def sh(*cmd, quiet=False):
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if (not quiet or r.returncode) and (r.stdout + r.stderr).strip():
        print((r.stdout + r.stderr).strip())
    if r.returncode:
        raise RuntimeError(f"failed: {' '.join(str(c) for c in cmd)}")
    return r.stdout

if not os.path.exists("matchday"):
    sh("git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO, "matchday")
if not os.path.exists("source"):
    sh("git", "init", "-q", "source")
    sh("git", "-C", "source", "fetch", "-q", "--depth", "1", REPO, "source-data" if LATEST else SOURCE_COMMIT)
    sh("git", "-C", "source", "checkout", "-q", "FETCH_HEAD")
PIPE = "matchday/analysis/pipeline"
sys.path.insert(0, PIPE)
sh(sys.executable, "-m", "pip", "install", "-q", "jax", "scipy", "pandas", "matplotlib", quiet=True)
print("code:", sh("git", "-C", "matchday", "log", "-1", "--format=%h %s", quiet=True).strip())
print("source data:", sh("git", "-C", "source", "log", "-1", "--format=%h %s", quiet=True).strip())

## 1. The published run

`data/season.json` carries everything the simulation needs: the table after the last completed matchday, the remaining fixtures, both sets of ratings and the doubt σ. Playing it again with a different seed should land within simulation noise of the published chances (about ±0.3 points for a 50% chance at 50,000 seasons).

In [ ]:
import numpy as np, pandas as pd
import season_sim as ss
site = json.load(open("matchday/data/season.json"))
ids = list(site["teams"])
names = {i: site["teams"][i]["short"] for i in ids}
table0 = {i: tuple(site["teams"][i]["table"]) + (0,) for i in ids}
fixtures = [(str(f["home"]), str(f["away"])) for f in site["fixtures"]]
rows = []
for view in ("models", "market"):
    v = site["views"][view]
    rt = ss.Ratings(att=v["ratings"]["att"], de=v["ratings"]["de"], mu=v["ratings"]["mu"], home=v["ratings"]["home"], rho=v["ratings"]["rho"])
    res = ss.simulate(ids, table0, fixtures, rt, v["sigma"], n=50_000, seed=1)
    for k, i in enumerate(ids):
        p = ss.outcome_probs(res.pos[k])
        rows.append(dict(view=view, club=names[i], title=p["title"], published_title=v["teams"][i]["p"]["title"],
                         drop=p["drop"], published_drop=v["teams"][i]["p"]["drop"]))
replay = pd.DataFrame(rows)
print(f"after matchday {site['afterMatchday']}, {len(fixtures)} matches to play")
replay[replay.view == "models"].sort_values("title", ascending=False).head(6).round(3)

In [ ]:
print("largest difference from the published numbers:",
      round(float((replay["title"] - replay["published_title"]).abs().max()), 4), "(title),",
      round(float((replay["drop"] - replay["published_drop"]).abs().max()), 4), "(relegation)")

## 2. Ratings from scratch

Build the match table from the source files, take the fixture list and today's results from OpenLigaDB, fit both sets of ratings on everything played before the cut-off and simulate. The bookmakers' ratings are fitted to the margin-free 1X2 odds of eight seasons of matches; my models' are the published Dixon–Coles (here without xG, see the note at the top).

In [ ]:
sh(sys.executable, f"{PIPE}/build_matches.py", "source", "matches.csv", quiet=True)
matches = pd.read_csv("matches.csv", parse_dates=["date"])
from site_odds import openligadb_season
season = int(site["season"])
ol = ss.openligadb_matches(openligadb_season(season))
md_done = ss.completed_matchday(ol, pd.Timestamp.now(tz="UTC"))
club = ss.club_names(ol, matches, season)
snap, teams, table0, fx, views = ss.run_snapshot(ol, matches, club, season, md_done, 20_000)
print(f"after matchday {md_done}; ratings from matches before {snap.cut:%d %b %Y}; "
      f"{snap.filled} played match(es) not in the source files yet (goals only)")
out = []
for view, (rt, res) in views.items():
    for k, t in enumerate(teams):
        s = ss.team_summary(res, k)
        out.append(dict(view=view, club=t, points=table0[t][0], expected=round(s["xpts"], 1), **{o: round(s["p"][o], 3) for o in ss.OUTCOME_KEYS}))
fresh = pd.DataFrame(out)
fresh[fresh.view == "market"].sort_values("expected", ascending=False)

In [ ]:
import matplotlib.pyplot as plt
models = fresh[fresh.view == "models"].set_index("club")
market = fresh[fresh.view == "market"].set_index("club")
order = market.sort_values("title", ascending=True).index[-6:]
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.barh(order, models.loc[order, "title"], color="#16161c", label="My models (goals only here)")
ax.scatter(market.loc[order, "title"], order, marker="|", s=400, color="#e02b3b", label="Bookmakers' ratings")
ax.set_xlim(0, 1); ax.set_xlabel("Title chance"); ax.legend(frameon=False, loc="lower right")
ax.set_title("Who wins the league?", loc="left"); plt.show()

## 3. What if?

Lock in results and play the rest. Here: the next matchday's first fixture goes to the away side. `fixed` takes exact scores; the page also takes plain win / draw / win picks.

In [ ]:
rt, base = views["models"]
first = 0
h, a = fx[first]
print(f"what if {a} win at {h}?")
after = ss.simulate(teams, table0, fx, rt, ss.SIGMA["models"], n=20_000, seed=1, fixed={first: (0, 1)})
before = ss.simulate(teams, table0, fx, rt, ss.SIGMA["models"], n=20_000, seed=1)
change = []
for k, t in enumerate(teams):
    b, f = ss.outcome_probs(before.pos[k]), ss.outcome_probs(after.pos[k])
    for o in ss.OUTCOME_KEYS:
        if abs(f[o] - b[o]) >= 0.01:
            change.append(dict(club=t, outcome=o, before=round(b[o], 3), after=round(f[o], 3)))
pd.DataFrame(change)

## 4. The test

The [pre-registered test](https://github.com/JHvL88/matchday/blob/main/analysis/backtest/season/PREREGISTRATION.md) replayed 2014/15 to 2025/26 from 34 points in time each. The settings (the doubt σ and the market's time decay) were chosen on the first five seasons; the last seven graded the result. The score is the positional RPS: how far the forecast over all 18 places was from where the club finished, 0 is perfect.

In [ ]:
res = json.load(open("matchday/analysis/backtest/season/results.json"))
t = res["test"]
pd.DataFrame({k: {"positional RPS": round(t[k]["overall"], 4), **{f"{int(s)}/{(int(s)+1)%100:02d}": round(v, 4) for s, v in t[k]["seasons"].items()}}
              for k in ("models", "market", "models_nodoubt", "market_nodoubt", "table", "uniform")}).T

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 4.4))
ax.plot([0, 1], [0, 1], color="#9a9aa4", linestyle="--", linewidth=1)
for key, face in (("market", "white"), ("models", "#16161c")):
    rows = t[key]["calibration"]
    ax.scatter([r["forecast"] for r in rows], [r["happened"] for r in rows], s=46, color=face, edgecolor="#16161c", label=key, zorder=3)
ax.set_xlabel("Forecast chance"); ax.set_ylabel("How often it happened"); ax.legend(frameon=False)
ax.set_title("Calibration, test seasons", loc="left"); plt.show()
print(res["verdict"])

### Check one test season

The bookmakers' ratings need only public data, so one test season can be recomputed exactly as the test ran it: the same checkpoints, settings, number of seasons and seeds. About two minutes on a CPU. Small differences in the last digit can come from a different processor (the rating fit is floating point); anything bigger means something changed.

In [ ]:
import season_backtest as sb
SEASON = 2022                      # 2022/23: Dortmund at 80% before the final day
ch = res["chosen"]["market"]
cur = sb.season_table(matches, SEASON)
clubs = sorted(set(cur.home) | set(cur.away))
final, _ = sb.final_positions(cur, clubs)
y = np.array([final[c] for c in clubs])
scores = []
for k in sb.CHECKPOINTS:
    cut, played, remaining = sb.checkpoint(cur, k)
    models_k = ss.model_ratings(matches, cut, SEASON, xg=False)      # only its rho is used here
    rt = ss.market_ratings(matches, cut, SEASON, models_k.rho, xi=ch["xi"])
    out = ss.simulate(clubs, ss.standings(played, clubs), list(zip(remaining.home, remaining.away)), rt, ch["sigma"],
                      n=sb.SIMS, seed=SEASON * 100 + k)
    scores.append(sb.rps_positions(out.pos / out.n, y).mean())
print(f"{SEASON}/{(SEASON+1)%100:02d}, bookmakers' ratings: recomputed {np.mean(scores):.4f}, report {t['market']['seasons'][str(SEASON)]:.4f}")

*Note on the last cell:* in the test, the market fit took its low-score parameter rho from my models' fit with xG. Without the private xG files rho comes from the goals-only fit here, so expect agreement to about three decimals rather than four.